# YouTube Transcript Fetcher

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook fetches transcripts from YouTube videos and exports them as structured data for qualitative analysis. Enter a YouTube URL or video ID, select your preferred language, and download the transcript as CSV, Excel, SRT, VTT, or plain text.

YouTube transcripts include both human-created captions and auto-generated captions. This notebook prioritizes human-created captions when available and falls back to auto-generated ones. No API key is required — the notebook uses the open-source `youtube-transcript-api` library to communicate directly with YouTube.

## Key Features

1. **No API Key Required**: Fetches transcripts directly from YouTube using open-source tools
2. **Language Selection**: Specify preferred languages with automatic fallback
3. **Caption Priority**: Prioritizes human-created captions over auto-generated ones
4. **Segment Chunking**: Merge short caption segments into longer chunks by time window
5. **Multiple Export Formats**: CSV, Excel, SRT, VTT, and plain text
6. **Whitespace Normalization**: Clean up caption text for tidy exports

## Workflow

1. **Configure**: Enter a YouTube URL or video ID and set language preferences
2. **Fetch**: Retrieve the transcript from YouTube
3. **Review**: Preview the transcript data in a table
4. **Export**: Download in your preferred format for further analysis

## Applications

This notebook supports research that involves analyzing video content — collecting interview transcripts from recorded sessions, gathering public discourse from lectures and presentations, or building datasets from video-based media for qualitative coding. The structured exports are formatted for use with qualitative analysis software and other tools in the AI Anthropology Toolkit.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using programmatic data retrieval to support qualitative research workflows. The notebook fetches and structures transcript data but does not interpret it. Analytical judgment remains with the researcher.

**Important**: Respect copyright and terms of service when using video transcripts. This notebook is intended for research purposes with appropriate permissions.

## Target Audience

Designed for anthropologists and qualitative researchers who need to collect and structure video transcript data — from graduate students analyzing recorded interviews to research teams building media corpora.

## Technical Approach

The notebook uses the **youtube-transcript-api** library to retrieve caption data directly from YouTube. It supports language preference ordering, human vs. auto-generated caption selection, and multiple subtitle export formats. All processing runs locally in the notebook.

## License & Attribution

This notebook is released under the **CC BY-NC 4.0** license. You may adapt and share it for non-commercial purposes with proper attribution.

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

Install required Python packages and import libraries for YouTube transcript retrieval, data processing, and interactive configuration. Run this cell first to ensure all dependencies are available.

In [ ]:
# Install required packages
!pip install youtube-transcript-api ipywidgets pandas openpyxl python-slugify -q

import re
import json
from datetime import timedelta

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from slugify import slugify

from youtube_transcript_api import (
    YouTubeTranscriptApi,
    TranscriptsDisabled,
    NoTranscriptFound,
    CouldNotRetrieveTranscript,
)

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print("\u2713 All packages installed and libraries loaded successfully")
print(f"\u2713 Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print("\U0001f4cb Ready to configure your transcript fetch!")

## Helper Functions

Core functions for parsing YouTube URLs, fetching transcripts, formatting timestamps, chunking segments, and exporting in multiple formats.

In [ ]:
# Helper Functions

YOUTUBE_REGEX = re.compile(
    r"(?:https?://)?(?:www\.|m\.)?youtube\.com/watch\?v=([\w-]{11})|(?:https?://)?youtu\.be/([\w-]{11})"
)

def parse_video_id(url_or_id):
    """Extract YouTube video ID from URL or raw ID."""
    url_or_id = url_or_id.strip()
    m = YOUTUBE_REGEX.search(url_or_id)
    if m:
        return m.group(1) or m.group(2)
    if re.fullmatch(r"[\w-]{11}", url_or_id):
        return url_or_id
    raise ValueError("Could not parse a valid YouTube video ID from the input.")


def normalize_text(t):
    """Collapse whitespace and trim text."""
    return re.sub(r"\s+", " ", t).strip()


def seconds_to_srt_time(s):
    """Convert seconds to SRT timestamp format (HH:MM:SS,mmm)."""
    ms = int(round((s - int(s)) * 1000))
    total = int(s)
    return f"{total // 3600:02}:{(total % 3600) // 60:02}:{total % 60:02},{ms:03}"


def seconds_to_vtt_time(s):
    """Convert seconds to VTT timestamp format (HH:MM:SS.mmm)."""
    ms = int(round((s - int(s)) * 1000))
    total = int(s)
    return f"{total // 3600:02}:{(total % 3600) // 60:02}:{total % 60:02}.{ms:03}"


def fetch_transcript(video_id, preferred_langs, allow_auto=True):
    """Fetch transcript from YouTube, prioritizing human-created captions."""
    api = YouTubeTranscriptApi()
    transcript_list = api.list(video_id)

    # Try human-created captions in preferred languages first
    for t in transcript_list:
        if not t.is_generated and t.language_code in preferred_langs:
            snippets = t.fetch()
            return t.language_code, [{'text': s.text, 'start': s.start, 'duration': s.duration} for s in snippets]

    # Then any human-created caption
    for t in transcript_list:
        if not t.is_generated:
            snippets = t.fetch()
            return t.language_code, [{'text': s.text, 'start': s.start, 'duration': s.duration} for s in snippets]

    # Fallback to auto-generated in preferred languages
    if allow_auto:
        for t in transcript_list:
            if t.is_generated and t.language_code in preferred_langs:
                snippets = t.fetch()
                return t.language_code, [{'text': s.text, 'start': s.start, 'duration': s.duration} for s in snippets]

        # Any auto-generated
        for t in transcript_list:
            if t.is_generated:
                snippets = t.fetch()
                return t.language_code, [{'text': s.text, 'start': s.start, 'duration': s.duration} for s in snippets]

    raise NoTranscriptFound("No transcript found in requested languages.")


def segments_to_dataframe(segments, normalize_whitespace=True):
    """Convert transcript segments to a pandas DataFrame."""
    df = pd.DataFrame(segments)
    if normalize_whitespace:
        df['text'] = df['text'].astype(str).map(normalize_text)
    df['end'] = df['start'] + df['duration']
    df.insert(0, 'idx', range(1, len(df) + 1))
    return df[['idx', 'start', 'end', 'duration', 'text']]


def chunk_segments(segments, chunk_seconds):
    """Merge adjacent segments into chunks of approximately chunk_seconds length."""
    if chunk_seconds <= 0:
        return segments
    out, cur, cur_start, cur_dur = [], [], None, 0.0
    for seg in segments:
        if not cur:
            cur_start = seg['start']
        cur.append(seg['text'])
        cur_dur += seg['duration']
        if cur_dur >= chunk_seconds:
            out.append({'text': ' '.join(cur), 'start': float(cur_start), 'duration': float(cur_dur)})
            cur, cur_start, cur_dur = [], None, 0.0
    if cur:
        out.append({'text': ' '.join(cur), 'start': float(cur_start or 0.0), 'duration': float(cur_dur)})
    return out


def export_srt(df):
    """Export DataFrame as SRT subtitle format."""
    lines = []
    for _, row in df.iterrows():
        lines.append(str(int(row['idx'])))
        lines.append(f"{seconds_to_srt_time(row['start'])} --> {seconds_to_srt_time(row['end'])}")
        lines.append(str(row['text']))
        lines.append("")
    return "\n".join(lines)


def export_vtt(df):
    """Export DataFrame as WebVTT subtitle format."""
    lines = ["WEBVTT", ""]
    for _, row in df.iterrows():
        lines.append(f"{seconds_to_vtt_time(row['start'])} --> {seconds_to_vtt_time(row['end'])}")
        lines.append(str(row['text']))
        lines.append("")
    return "\n".join(lines)


print("\u2713 Helper functions defined")

## Transcript Fetch Interface

Enter a YouTube URL or video ID, configure language and export preferences, then fetch the transcript.

In [ ]:
# Interactive Fetch Interface

style = {'description_width': '120px'}
layout = widgets.Layout(width='500px')

instructions_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
instructions_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f3ac YouTube Transcript Fetcher</h2>'
instructions_html += '<p><strong>Welcome!</strong> This notebook fetches transcripts from YouTube videos for qualitative research.</p>'
instructions_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
instructions_html += '<ol>'
instructions_html += '<li><strong>Enter:</strong> Paste a YouTube URL or video ID below</li>'
instructions_html += '<li><strong>Configure:</strong> Set language and export preferences</li>'
instructions_html += '<li><strong>Fetch:</strong> Click the button to retrieve the transcript</li>'
instructions_html += '<li><strong>Export:</strong> File downloads automatically</li>'
instructions_html += '</ol>'
instructions_html += '<p style="margin-top: 10px; color: #8B8C89; font-size: 0.9em;">'
instructions_html += 'Human-created captions are prioritized. Auto-generated captions are used as fallback.</p>'
instructions_html += '</div>'

instructions = widgets.HTML(value=instructions_html)

video_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f3ac Video</h3>')

video_input = widgets.Text(
    value='',
    placeholder='e.g. https://youtu.be/VIDEOID or just the 11-character ID',
    description='YouTube URL:',
    layout=widgets.Layout(width='500px'),
    style=style
)

config_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\u2699\ufe0f Options</h3>')

langs_input = widgets.Text(
    value='en',
    placeholder='Comma-separated, e.g. en,es,fr',
    description='Languages:',
    layout=layout,
    style=style
)

langs_help = widgets.HTML(
    '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
    'Enter preferred languages in order. The notebook tries each in sequence.</p>'
)

allow_auto_chk = widgets.Checkbox(
    value=True,
    description='Allow auto-generated captions as fallback',
    indent=False,
    style={'description_width': 'auto'}
)

chunk_slider = widgets.IntSlider(
    value=0, min=0, max=600, step=5,
    description='Chunk (sec):',
    style=style
)

chunk_help = widgets.HTML(
    '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
    'Set to 0 for original segments. Set higher to merge segments into longer chunks.</p>'
)

normalize_chk = widgets.Checkbox(
    value=True,
    description='Normalize whitespace',
    indent=False,
    style={'description_width': 'auto'}
)

export_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c1 Export</h3>')

format_dd = widgets.Dropdown(
    options=[('CSV (.csv)', 'csv'), ('Excel (.xlsx)', 'xlsx'), ('SRT (.srt)', 'srt'),
             ('VTT (.vtt)', 'vtt'), ('Plain Text (.txt)', 'txt')],
    value='csv',
    description='Format:',
    layout=layout,
    style=style
)

fetch_btn = widgets.Button(
    description='Fetch Transcript',
    button_style='',
    layout=widgets.Layout(width='200px', margin='20px 0'),
    style={'button_color': '#6096BA'}
)

out = widgets.Output()


def on_fetch_clicked(_):
    out.clear_output()
    with out:
        try:
            vid = parse_video_id(video_input.value)
            langs = [x.strip() for x in langs_input.value.split(',') if x.strip()]
            if not langs:
                langs = ['en']

            print(f"\U0001f50d Fetching transcript for video: {vid}")
            print(f"   Languages: {", ".join(langs)}")
            print()

            lang_code, segments = fetch_transcript(vid, langs, allow_auto=allow_auto_chk.value)
            print(f"\u2713 Found transcript in: {lang_code} ({len(segments)} segments)")

            if chunk_slider.value > 0:
                segments = chunk_segments(segments, chunk_slider.value)
                print(f"\u2713 Chunked into {len(segments)} segments ({chunk_slider.value}s windows)")

            df = segments_to_dataframe(segments, normalize_whitespace=normalize_chk.value)
            print()
            print("\U0001f4ca Preview:")
            display(df.head(10))

            # Export
            base = slugify(f'{vid}_{lang_code}')
            fmt = format_dd.value

            if fmt == 'xlsx':
                filepath = f'{base}.xlsx'
                df.to_excel(filepath, sheet_name='Transcript', index=False, engine='openpyxl')

                from openpyxl import load_workbook
                from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
                wb = load_workbook(filepath)
                header_fill = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
                header_font = Font(bold=True, color='FFFFFF')
                thin_border = Border(
                    left=Side(style='thin', color='E7ECEF'), right=Side(style='thin', color='E7ECEF'),
                    top=Side(style='thin', color='E7ECEF'), bottom=Side(style='thin', color='E7ECEF'),
                )
                for ws in wb.worksheets:
                    for cell in ws[1]:
                        cell.fill = header_fill
                        cell.font = header_font
                        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
                        cell.border = thin_border
                    for col in ws.columns:
                        max_len = max(len(str(c.value or '')) for c in col)
                        ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 10), 60)
                    ws.freeze_panes = 'A2'
                    for row in ws.iter_rows(min_row=2):
                        for cell in row:
                            cell.alignment = Alignment(vertical='top', wrap_text=True)
                            cell.border = thin_border
                wb.save(filepath)
            elif fmt == 'csv':
                filepath = f'{base}.csv'
                df.to_csv(filepath, index=False)
            elif fmt == 'srt':
                filepath = f'{base}.srt'
                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(export_srt(df))
            elif fmt == 'vtt':
                filepath = f'{base}.vtt'
                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(export_vtt(df))
            elif fmt == 'txt':
                filepath = f'{base}.txt'
                with open(filepath, 'w', encoding='utf-8') as f:
                    for _, row in df.iterrows():
                        f.write(str(row['text']) + '\n')

            print()
            print(f"\u2713 Saved: {filepath}")

            if IN_COLAB:
                colab_files.download(filepath)
                print(f"\U0001f4e5 Downloaded: {filepath}")

        except TranscriptsDisabled:
            print("\u26a0\ufe0f Transcripts are disabled for this video.")
        except NoTranscriptFound:
            print("\u26a0\ufe0f No transcript found in the requested languages.")
        except CouldNotRetrieveTranscript as e:
            print(f"\u26a0\ufe0f Could not retrieve transcript: {e}")
        except ValueError as e:
            print(f"\u26a0\ufe0f Input error: {e}")
        except Exception as e:
            print(f"\u274c Error: {type(e).__name__}: {e}")


fetch_btn.on_click(on_fetch_clicked)

display(widgets.VBox([
    instructions,
    video_header,
    video_input,
    config_header,
    langs_input,
    langs_help,
    allow_auto_chk,
    chunk_slider,
    chunk_help,
    normalize_chk,
    export_header,
    format_dd,
    fetch_btn,
    out,
]))